In [5]:
import os
import numpy as np
import cv2
from deepface import DeepFace
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

# === 1. Définir les paramètres ===
DATASET_PATH = "/kaggle/input/dataset-org/Dataset_resized"  # Dossier contenant les images des visages organisées par classe
MODEL_NAME = "ArcFace"  # Choix parmi ["VGG-Face", "Facenet", "Facenet512", "OpenFace", "DeepFace", "ArcFace", "Dlib"]
OUTPUT_MODEL = "face_model.pkl"  # Fichier pour sauvegarder le modèle entraîné

# === 2. Charger les images et extraire les embeddings ===
def load_data(dataset_path, model_name):
    X, y = [], []
    for label in os.listdir(dataset_path):  # Parcourir les dossiers des classes (noms des personnes)
        person_dir = os.path.join(dataset_path, label)
        if os.path.isdir(person_dir):
            print(f"Extraction commencé pour {label}•••")
            for img_name in os.listdir(person_dir):
                img_path = os.path.join(person_dir, img_name)
                try:
                    # Lire l'image
                    img = cv2.imread(img_path)
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                    # Extraire les embeddings avec DeepFace
                    embedding = DeepFace.represent(img_path=img_path, model_name=model_name, enforce_detection=False)[0]["embedding"]
                    X.append(embedding)
                    y.append(label)
                except Exception as e:
                    print(f"Erreur avec {img_path}: {e}")
        print(f"Extraction terminé pour {label}♦♦♦")
    
    return np.array(X), np.array(y)

print("🔍 Chargement des images et extraction des embeddings...")
X, y = load_data(DATASET_PATH, MODEL_NAME)

# === 3. Encoder les labels ===
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# === 4. Entraîner un classificateur SVM ===
print("🚀 Entraînement du modèle SVM...")
svm_model = make_pipeline(StandardScaler(), SVC(kernel='linear', probability=True))
svm_model.fit(X, y_encoded)

# === 5. Sauvegarde du modèle et des labels ===
joblib.dump({"model": svm_model, "encoder": label_encoder, "deepface_model": MODEL_NAME}, OUTPUT_MODEL)
print(f"Modèle enregistré sous '{OUTPUT_MODEL}'")

🔍 Chargement des images et extraction des embeddings...
Extraction commencé pour AKODEGNON_Crepin•••
Extraction terminé pour AKODEGNON_Crepin♦♦♦
Extraction commencé pour Martine_TOVIESSI•••
Extraction terminé pour Martine_TOVIESSI♦♦♦
Extraction commencé pour TCHABI_Hubert•••
Extraction terminé pour TCHABI_Hubert♦♦♦
Extraction commencé pour GBEDOAGO_Alexandre•••
Extraction terminé pour GBEDOAGO_Alexandre♦♦♦
Extraction commencé pour IDRISSOU_Issahou•••
Extraction terminé pour IDRISSOU_Issahou♦♦♦
Extraction commencé pour MAHOUTIN_Salem•••
Extraction terminé pour MAHOUTIN_Salem♦♦♦
Extraction commencé pour KPANOU_Finagnon_Affissou•••
Extraction terminé pour KPANOU_Finagnon_Affissou♦♦♦
Extraction commencé pour SEDJIDE_Patience•••
Extraction terminé pour SEDJIDE_Patience♦♦♦
Extraction commencé pour TOKO_Boni_Orou•••
Extraction terminé pour TOKO_Boni_Orou♦♦♦
Extraction commencé pour NOMBIME_Rufin•••
Extraction terminé pour NOMBIME_Rufin♦♦♦
Extraction commencé pour FAMBO_Bruno•••
Extraction ter

In [4]:
# === 6. Chargement et prédiction sur une nouvelle image ===
import joblib
from deepface import DeepFace
import numpy as np

# Charger le modèle enregistré
model_data = joblib.load("face_model.pkl")
svm_model = model_data["model"]
label_encoder = model_data["encoder"]
deepface_model = model_data["deepface_model"]

# Charger une nouvelle image et extraire son embedding
img_path = "img/img_001.jpg"
embedding = DeepFace.represent(img_path=img_path, model_name=deepface_model)[0]["embedding"]

# Prédire l'identité
pred_label = svm_model.predict([embedding])[0]
pred_name = label_encoder.inverse_transform([pred_label])[0]

print(f"🧐 La personne sur l'image est : {pred_name}")

🧐 La personne sur l'image est : OLAOGOU_Carlos_Martial


In [5]:
# === 7. Prédiction en temps réel(webcam) et enregistrement de la présence dans un fichier Excel ===
import os
import cv2
import face_recognition
from deepface import DeepFace
import numpy as np
import joblib
from datetime import datetime
from openpyxl import Workbook, load_workbook
import hashlib

# Charger les modèles
model_data = joblib.load("face_model.pkl")
svm_model = model_data["model"]
label_encoder = model_data["encoder"]
deepface_model = model_data["deepface_model"]

# Capture webcam
cap = cv2.VideoCapture(0) 

date = datetime.now()
date_aujourdhui = date.strftime("%d-%m-%Y")

fichier_excel = "Liste_de_présence.xlsx"
try:
    if os.path.exists(fichier_excel):
        df = load_workbook(fichier_excel)
    else:
        df = Workbook()
        df.remove(df.active)

    if date_aujourdhui not in df.sheetnames:
        feuille = df.create_sheet(date_aujourdhui)
        feuille.append(["Noms et Prénoms", "Heures", "Dates"])
    else:
        feuille = df[date_aujourdhui]

    personnes_deja_presentes = set(ligne[0] for ligne in feuille.iter_rows(min_row=2, values_only=True) if ligne[0])
    visages_reconnus = {}  # cache des visages reconnus

    print("\n📸 Webcam en cours... Appuyez sur 'q' pour quitter.\n")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

        face_locations = face_recognition.face_locations(rgb_small_frame)

        for top, right, bottom, left in face_locations:
            top *= 4
            right *= 4
            bottom *= 4
            left *= 4

            face_img = frame[top:bottom, left:right]
            face_id = hashlib.md5(face_img.tobytes()).hexdigest()

            if face_id in visages_reconnus:
                name = visages_reconnus[face_id]["name"]
                confidence = visages_reconnus[face_id]["confidence"]
            else:
                try:
                    embedding = DeepFace.represent(
                        img_path=face_img,
                        model_name=deepface_model,
                        enforce_detection=False
                    )[0]["embedding"]

                    proba = svm_model.predict_proba([embedding])[0]
                    pred_label = np.argmax(proba)
                    confidence = np.max(proba)

                    if confidence >= 0.90:
                        name = label_encoder.inverse_transform([pred_label])[0]
                        visages_reconnus[face_id] = {"name": name, "confidence": confidence}

                        if name not in personnes_deja_presentes:
                            maintenant = datetime.now()
                            heure = maintenant.strftime("%H:%M:%S")
                            date = maintenant.strftime("%d/%m/%Y")
                            feuille.append([name, heure, date])
                            df.save(fichier_excel)
                            personnes_deja_presentes.add(name)
                            print(f"✅ Présence enregistrée : {name} à {heure} le {date}")
                    else:
                        name = "Inconnu"
                except Exception:
                    name = "Inconnu"
                    confidence = 0

            # Affichage
            cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0) if name != "Inconnu" else (0, 0, 255), 2)
            cv2.putText(
                frame, f"{name} ({confidence:.2f})",
                (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                (0, 255, 0) if name != "Inconnu" else (0, 0, 255), 2
            )

        cv2.imshow("Reconnaissance Faciale", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

except Exception as e:
    print("Erreur pendant l'enregistrement: ", e)

finally:
    df.close()
    print("\n📁 Session terminée, fichier sauvegardé !")


📸 Webcam en cours... Appuyez sur 'q' pour quitter.

✅ Présence enregistrée : OLAOGOU_Carlos_Martial à 12:33:29 le 07/06/2025

📁 Session terminée, fichier sauvegardé !
